In [6]:
import requests
import pandas as pd
from datetime import datetime
from google.transit import gtfs_realtime_pb2



URL = "https://realtime.gtfs.de/realtime-free.pb"

response = requests.get(URL, timeout=30)

print(response.status_code)
print(len(response.content), "bytes")


200
11541722 bytes


In [ ]:

feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(response.content)

print(f"Number of entities: {len(feed.entity)}")


Number of entities: 44253


In [3]:
for entity in feed.entity[:10]:
    print(entity)

id: "162015tu"
trip_update {
  trip {
    trip_id: "162015"
    start_date: "20260904"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 0
    departure {
      delay: 0
      time: 1788559500
    }
    stop_id: "614810"
    schedule_relationship: SCHEDULED
  }
}

id: "912852tu"
trip_update {
  trip {
    trip_id: "912852"
    start_date: "20260904"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 1
    arrival {
      delay: 0
      time: 1788560100
    }
    departure {
      delay: 0
      time: 1788560100
    }
    stop_id: "606206"
    schedule_relationship: SCHEDULED
  }
}

id: "477614tu"
trip_update {
  trip {
    trip_id: "477614"
    start_date: "20260904"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 0
    departure {
      delay: 0
      time: 1788561300
    }
    stop_id: "682037"
    schedule_relationship: SCHEDULED
  }
}

id: "1646745tu"
trip_update {
  trip {
    trip_id: "1

In [10]:
stops_df = pd.read_csv("../data/mvv_stops.csv", delimiter=";")


In [17]:
import pandas as pd
from datetime import datetime

stops_df["HstNummer"] = stops_df["HstNummer"].astype(str)
df["stop_id"] = df["stop_id"].astype(str)

# Create stop_id -> stop name mapping
stop_names = stops_df.set_index("HstNummer")["Name ohne Ort"].to_dict()

rows = []

for entity in feed.entity:
    if not entity.HasField("trip_update"):
        continue

    trip = entity.trip_update.trip

    for stop in entity.trip_update.stop_time_update:

        row = {
            "trip_id": trip.trip_id,
            "start_date": trip.start_date,
            "stop_id": stop.stop_id,
            "stop_name": stop_names[stop.stop_id],
            "stop_sequence": stop.stop_sequence,
        }

        if stop.HasField("arrival"):
            row["arrival_time"] = datetime.fromtimestamp(
                stop.arrival.time
            )
            row["arrival_delay"] = stop.arrival.delay

        if stop.HasField("departure"):
            row["departure_time"] = datetime.fromtimestamp(
                stop.departure.time
            )
            row["departure_delay"] = stop.departure.delay

        rows.append(row)

df = pd.DataFrame(rows)

df.head(100)

KeyError: '614810'

In [16]:
stop_names

{'1': 'Karlsplatz (Stachus)',
 '2': 'Marienplatz',
 '3': 'Isartor',
 '4': 'Rosenheimer Platz',
 '5': 'Ostbahnhof',
 '6': 'Hauptbahnhof (S, U, Bus, Tram)',
 '7': 'Hackerbrücke',
 '8': 'Donnersbergerbrücke',
 '9': 'Laim',
 '10': 'Pasing',
 '11': 'Leonrodplatz',
 '12': 'Hochschule München (Lothstr.)',
 '13': 'Sandstraße',
 '14': 'Karlspl.(Stachus) Nord',
 '15': 'Karlstraße',
 '16': 'Lenbachplatz',
 '17': 'Nationaltheater',
 '18': 'Kammerspiele',
 '19': 'Maxmonument',
 '20': 'Marienplatz (Theatinerstraße)',
 '21': 'Maximilianeum',
 '22': 'Flurstraße',
 '23': 'Grillparzerstraße',
 '25': 'Alter Messeplatz',
 '26': 'Isartor/Zweibrückenstraße',
 '27': 'Ridlerstraße',
 '28': 'Rindermarkt',
 '29': 'Landshuter Allee',
 '30': 'Poccistraße',
 '31': 'Hirschgarten',
 '32': 'Hochschule München',
 '34': 'Nordbad',
 '35': 'Görresstraße',
 '36': 'Georgenstraße',
 '38': 'Müllerstraße',
 '40': 'Goetheplatz',
 '41': 'Eduard-Schmid-Straße',
 '42': 'Mariahilfplatz',
 '43': 'Kurfürstenplatz',
 '44': 'Winzerers

In [ ]:
df["arrival_time"].min()


Timestamp('2026-09-04 10:14:00')